<a href="https://colab.research.google.com/github/SY-256/llms-from-scratch/blob/main/notebooks/dpo-from-scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLMアライメントのための直接選好最適化（DPO）の実装

## DPO
- ユーザーの期待や指示に合ったレスポンスを生成するようにモデルをファインチューニング（アライメント）する手法
- 一方の回答スタイルを他方より好むようにLLMを教育する手法。ユーザーの選好にそって調整する
- 人間の選好または特定の目標に合わせてモデルの出力を直接最適化することに焦点を当てている

## 2.1 データセットの読み込み

In [ ]:
import json
import os
import requests

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)
    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    data = json.loads(text_data)
    return data


file_path = "instruction-data-with-preference.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/04_preference-tuning-with-dpo/instruction-data-with-preference.json"
)

data = download_and_load_file(file_path, url)
print(f"Number of entries: {len(data)}")

In [ ]:
# 2つのサンプルエントリを確認
import pprint

pprint.pp(data[50])

In [ ]:
pprint.pp(data[999])

`'chosen'`と`'rejected'`エントリはDPOに使用するもので、`'chose'`が選好された応答、`'rejected'`が非選好の応答

- ファインチューニングの目標は、モデルが拒否された応答ではなく、選好された応答のスタイルに従うようにすること

In [ ]:
# Alpacaプロンプトスタイルを適用してモデル入力をフォーマットするユーティリティ関数
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else " "

    return instruction_text + input_text

In [ ]:
# サンプルでフォーマット適用確認
model_input = format_input(data[50])
print(model_input)

In [ ]:
# Alpacaプロンプトスタイルを使用して、選好された応答と拒否された応答をフォーマットする
desired_response = f"### Response:\n{data[50]['chosen']}"
print(desired_response)

In [ ]:
possible_response = f"### Response:\n{data[50]['rejected']}"
print(possible_response)

## 2.2 学習・検証・テストデータの分割
- 学習データ:85%, 検証データ:5%、テストデータ:10%に分割

In [ ]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_portion = int(len(data) - train_portion - test_portion)

train_data = data[:train_portion]
val_data = data[train_portion+test_portion:]
test_data = data[train_portion:train_portion + test_portion]

In [ ]:
print(f"Training set length: {len(train_data)}")
print(f"Validation set length: {len(val_data)}")
print(f"Test set length: {len(test_data)}")

## 2.3 `PreferenceDataset`クラスとバッチ処理関数の開発
- 単一の出力シーケンス（応答）に注目する代わりに、一方が他方よりも選好（「chosen」）される応答ペアを返すようにデータセットクラスを修正する

In [ ]:
import torch
from torch.utils.data import Dataset

class PreferenceDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # テキストを事前にトークン化
        self.encoded_texts = []
        for entry in data:
            prompt = format_input(entry)
            rejected_response = entry["rejected"]
            chosen_response = entry["chosen"]

            prompt_tokens = tokenizer.encode(prompt)
            chosen_full_text = f"{prompt}\n\n### Response:\n{chosen_response}"
            rejected_full_text = f"{prompt}\n\n### Response:\n{rejected_response}"
            chosen_full_tokens = tokenizer.encode(chosen_full_text)
            rejected_full_tokens = tokenizer.encode(rejected_full_text)

            self.encoded_texts.append({
                "prompt": prompt_tokens,
                "chosen": chosen_full_tokens,
                "rejected": rejected_full_tokens,
            })

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

- 更新された`PreferenceDataset`クラスとともに、バッチ内の各シーケンスを同じ長さにパディングするために使用するバッチレコード関数も更新する必要がある。これにより、バッチとしてまとめることができる

In [ ]:
def custom_collate_fn(
        batch,
        pad_token_id=50256,
        allowed_max_length=None,
        mask_prompt_tokens=True,
        device="cpu"
):
    # バッチデータを保持するリストを初期化
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": [],
    }

    # 共通パディング長を設定するために最も長いシーケンスを確定
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max(len(item[key])+1 for item in batch)
            max_length_common = max(max_length_common, current_max)

    # バッチ内の各アイテムを処理
    for item in batch:
        prompt = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt)

        for key in ["chosen", "rejected"]:
            # 共通最大長に応じてパディングを調整
            sequence = item[key]
            padded = sequence + [pad_token_id] * (max_length_common - len(sequence))
            mask = torch.ones(len(padded)).bool()

            # すべてのパディングトークンのマスクをFalseに設定
            mask[len(sequence):] = False

            # すべての入力トークンのマスクをFalseに設定
            # +2 は"### Response" の前の2つ改行("\n")トークンをFalseに設定
            if mask_prompt_tokens:
                mask[:prompt.shape[0]+2] = False

            batch_data[key].append(torch.tensor(padded))
            batch_data[f"{key}_mask"].append(mask)

    # 最終処理
    for key in ["chosen", "rejected", "chosen_mask", "rejected_mask"]:
        # 指定されたキーのすべてのシーケンスをテンソルにスタック
        tensor_stack = torch.stack(batch_data[key])

        # オプションで最大シーケンス長に切り詰め
        if allowed_max_length is not None:
            tensor_stack = tensor_stack[:, :allowed_max_length]

        # 指定されたデバイスに移動
        batch_data[key] = tensor_stack.to(device)

    return batch_data

In [ ]:
# サンプルで確認
from functools import partial

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    mask_prompt_tokens=True,
    allowed_max_length=1024 # モデルがサポートするコンテキストサイズの最大を設定
)

In [ ]:
# サンプルで確認
example_data = data[:2]

for i in example_data:
    print()
    pprint.pp(i)

In [ ]:
import tiktoken
from torch.utils.data import DataLoader

tokenizer = tiktoken.get_encoding("gpt2")

example_dataset = PreferenceDataset(example_data, tokenizer)

example_dataloader = DataLoader(
    example_dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False
)

In [ ]:
# データセットのキーを確認
for batch in example_dataloader:
    break

print(f"batch.keys: {batch.keys()}")

In [ ]:
# promptはテンソルのリストで、各テンソルには指定されたサンプルのトークンIDが含まれている
# バッチサイズは2を指定したので、2つのトークンIDテンソルのリストがある
batch["prompt"]

In [ ]:
# chosenとrejectedの応答エントリはテンソルとしてスタックできるようにパディングされている
batch["chosen"]

In [ ]:
# トークンIDからテキストに戻すためのユーティリティ関数
def decode_tokens_from_batch(token_ids, tokenizer):
    ids_in_python_list = token_ids.flatten().tolist()
    return tokenizer.decode(ids_in_python_list)

In [ ]:
# バッチ内の最初のプロンプトエントリに`decode_tokens_from_batch`ユーティリティ関数を適用
text = decode_tokens_from_batch(
    token_ids=batch["prompt"][0],
    tokenizer=tokenizer
)
print(text)

In [ ]:
# `chosen`に対しても同じように変換してみる
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0],
    tokenizer=tokenizer
)
print(text)

- `<|endoftext|>`のパディングトークンは損失計算で無視されるため、学習結果に影響しない

In [ ]:
# 同じように`rejected`についても確認
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0],
    tokenizer=tokenizer
)
print(text)

データマスクについて:
- 各データセットエントリに対して`"chosen_mask"`と`"rejected_mask"`を作成している
- マスクは`"chosen"`エントリに示すように、応答エントリと同じ形状を持つ：

In [ ]:
print(f"chosen inputs: {batch['chosen'][0].shape}")
print(f"chosen mask: {batch['chosen_mask'][0].shape}")

In [ ]:
# マスクエントリの内容は真理値(bool型)
batch["chosen_mask"][0]

- `True`値は実際の応答に対応するトークンIDを返す
- `False`トークンは、プロンプトトークン（`customized_collate_fn`関数で`mask_prompt_tokens=True`を設定した場合）またはパディングトークンに対応するトークンIDを示す
- マスクを選択マスクとして使用して、応答に対する（`True`の）トークンIDのみを選択できる。つまり、すべてのプロンプトとパディングトークンを取り除くことができる

In [ ]:
# 取り除けるか確認
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0][batch["chosen_mask"][0]], # TrueのものだけトークンIDが返ってくる
    tokenizer=tokenizer
)
print(text)

In [ ]:
text = decode_tokens_from_batch(
    token_ids=batch["rejected"][0][batch["rejected_mask"][0]],
    tokenizer=tokenizer
)
print(text)

- このマスクを使用して、後でDPO損失を計算する際にプロンプトとパディングトークンを無視する

## 2.4 学習・検証・テストセットのデータローダー作成

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = PreferenceDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [ ]:
val_dataset = PreferenceDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

test_dataset = PreferenceDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

In [ ]:
# データローダーからデータセットの形状を確認
for batch in train_loader:
    print(
        batch["chosen"].shape,
        batch["rejected"].shape
    )

- 各行は各バッチの`chosen`と`rejected`エントリの形状を示している
- バッチごとにパディングを適用しているため、各行の形状が異なる
- 効率化のためであり、データセット全体で最長のサンプルにすべてのサンプルをパディングする（全体で同じ長さになる）のは非効率であるから

In [ ]:
# Google Driveのモデルパス
from google.colab import drive
import shutil

drive.mount("/content/drive")
model_path = "/content/drive/MyDrive/colab-notebooks/LLMs-from-scratch/model/gpt2-medium355M-sft.pth"
shutil.copy(model_path, ".")

In [ ]:
! git clone https://github.com/rasbt/LLMs-from-scratch.git

In [ ]:
import sys
sys.path.append('/content/LLMs-from-scratch/ch07/04_preference-tuning-with-dpo')

In [ ]:
# GPTモデルの設定
from previous_chapters import GPTModel

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layesrs": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model = GPTModel(BASE_CONFIG)

In [ ]:
# ファインチューニングしたモデルの読み込み
model.load_state_dict(
    torch.load(
        "gpt2-medium355M-sft.pth",
        map_location=torch.device("cpu"),
        weights_only=True
    )
)
model.eval();

- ファインチューニングされたモデルが正しく保存・読み込まれていることを確認

In [ ]:
prompt = """Below is an instruction that describes a task. Write a response
that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'
"""

In [ ]:
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)

torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(prompt, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

response = token_ids_to_text(token_ids, tokenizer)
print(response)

In [ ]:
# 応答テキストのみを返すように応答の内容をクリーンアップする
def extract_response(response_text, input_text):
    # 入力の指示プロンプト全部削って、"### Response:"の記載も削る
    return response_text[len(input_text):].replace("### Response:", "").strip()

response = extract_response(response, prompt)
print(response)

- DPOは2つのLLMを使って動作する。
- ポリシーモデル（最適化したいLLM）と参照モデル（変更しない元のモデル）
- `model` を`policy_model`と`reference_model`として2つ用意する

In [ ]:
policy_model = model

reference_model = GPTModel(BASE_CONFIG)
reference_model.load_state_dict(
    torch.load(
        "gpt2-medium355M-sft.pth",
        map_location=torch.device("cpu"),
        weights_only=True
    )
)
reference_model.eval();

policy_model.to(device)
reference_model.to(device);

## DPO損失関数のコーディング

In [ ]:
# DPO損失
import torch.nn.functional as F

def compute_dpo_loss(
        model_chosen_lobprobs,
        model_rejected_logprobs,
        reference_chosen_logprobs,
        reference_rejected_logprobs,
        beta=0.1,
):
    model_logrations = model_chosen_lobprobs - model_rejected_logprobs
    reference_logprobs = reference_chosen_logprobs - reference_rejected_logprobs
    logits = model_logrations - reference_logprobs

    # DPO計算
    losses = -F.logsigmoid(beta * logits)

    # 学習の進捗を追跡するためのオプション
    chosen_rewards = (model_chosen_lobprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    # バッチ内のサンプル平均を取るための .mean()
    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()

In [ ]:
# 対数確率の計算（logprobs）
def compute_logprobs(logits, labels, selection_mask=None):
    # ラベルは1つシフトされた入力
    labels = labels[:, 1:].clone()

    # ラベルの num_tokens に合わせてロジットを切り詰め
    logits = logits[:, :-1, :]

    log_probs = F.log_softmax(logits, dim=-1)

    # 実際のラベルの対数確率を収集
    selected_log_probs = torch.gather(
        input=log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()

        # パディングトークンをフィルタリングするためにマスクを適用
        selected_log_probs = selected_log_probs * mask

        # パディングトークンを除いた平均対数確率を計算
        # トークン全体の平均なので、形状は（batch_size,）
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)

        return avg_log_prob

    else:
        return selected_log_probs.mean(-1)

In [ ]:
# torch.gather()関数の挙動確認
# cross_entropy関数の内部処理に挙動は似ている

# サンプルデータ
logits = torch.tensor(
    [[2.0, 1.0, 0.1],
     [0.5, 2.5, 0.3]] # Shape: (2, 3)
)
targets = torch.tensor([0, 2]) # Shape: (2,)

# torch.gather() を使った手動損失計算
log_softmax_logits = F.log_softmax(logits, dim=-1) # Shape: (2, 3)
selected_log_probs = torch.gather(
    input=log_softmax_logits,
    dim=1,
    index=targets.unsqueeze(1) # Shape: (2, 1) バッチ方向にunsqueeze(1)で軸追加
).squeeze(1) # Shape: (2,)
manual_loss = -selected_log_probs.mean() # バッチ全体での損失の平均

# Pytorchで計算したときの損失
cross_entropy = F.cross_entropy(logits, targets)

print(manual_loss, cross_entropy)

In [ ]:
# torch.gatherのメカニズムをもう少し詳しく
t = torch.tensor(
    [[1., 2.],
     [3., 4.]]
)

m = torch.tensor(
    [[1, 1],
     [0, 1]]
)

- `t`は選択したいテンソルで、`m`は選択方法を指定するマスク
- 例えば、`m`の最初の行に`[1, 1]`が含まれるため、`t`のインデックス位置`1`の値（値`2.`）を2回選択する
- `m`の2番目の行`[0, 1]`は、`t`の2番目の行のインデックス位置0と1を選択する。これは`3.`と`4.`

In [ ]:
torch.gather(input=t, dim=-1, index=m)

- `torch.gather()`は選択関数
- 50,257トークンの語彙内に正しいトークンに対応する対数確率を取得するために使用した
- 「正しい」トークンとは応答エントリーで与えられているトークン

- `cross_entropy`関数と本質的には同じであるが、`torch.gather`関数の方が少し多くの制御ができるため使用している
- `selection_mask`は、プロンプトとパディングトークンをオプションで無視するためのもの

In [ ]:
# インプットバッチに対するDPO損失計算
def compute_dpo_loss_bath(batch, policy_model, reference_model, beta):

    # policy_model(batch["chosen"])がロジット
    policy_chosen_log_probs = compute_logprobs(
        logits=policy_model(batch["chosen"]),
        labels=batch["chosen"],
        selection_mask=batch["chosen_mask"]
    )
    policy_rejected_log_probs = compute_logprobs(
        logits=policy_model(batch["rejected"]),
        labels=batch["rejected"],
        selection_mask=batch["rejected_mask"]
    )

    with torch.no_grad():
        ref_chosen_log_probas = compute_logprobs(
            logits=reference_model(batch["chosen"]),
            labels=batch["chosen"],
            selection_mask=batch["chosen_mask"]
        )
        ref_rejected_log_probs = compute_logprobs(
            logits=reference_model(batch["rejected"]),
            labels=batch["rejected"],
            selection_mask=batch["rejected_mask"]
        )

    loss, chosen_rewards, rejected_rewards = compute_dpo_loss(
        model_chosen_lobprobs=policy_chosen_log_probs,
        model_rejected_logprobs=policy_rejected_log_probs,
        reference_chosen_logprobs=ref_chosen_log_probas,
        reference_rejected_logprobs=ref_rejected_log_probs,
        beta=beta
    )
    return loss, chosen_rewards, rejected_rewards

In [ ]:
# 単一バッチに対して作動する
with torch.no_grad():
    loss = compute_dpo_loss_bath(batch, policy_model, reference_model, beta=0.1)
print(loss)

`num_batches`で動作するように拡張する

In [ ]:
def compute_dpo_loss_loader(data_loader, policy_model, reference_model, beta, num_batches=None):

    total_loss, total_chosen_rewards, total_rejected_rewards = 0., 0., 0.
    if len(data_loader) == 0:
        return float("nan")

    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # データローダーの総バッチ数に合わせてバッチ数を減らす
        # num_batches がデータローダーのバッチ数を超える場合
        num_batches = min(num_batches, len(data_loader))

    for i, batch in enumerate(data_loader):
        if i < num_batches:
            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_bath(
                batch=batch,
                policy_model=policy_model,
                reference_model=reference_model,
                beta=beta
            )
            total_loss += loss.item()
            total_chosen_rewards += chosen_rewards.item()
            total_rejected_rewards += rejected_rewards.item()

        else:
            break

    # 平均を計算
    total_loss /= num_batches
    total_chosen_rewards /= num_batches
    total_rejected_rewards /= num_batches

    return total_loss, total_chosen_rewards, total_rejected_rewards

- `num_batches`を指定する理由 -> 効率化のため（毎回データセット全体の損失を計算すると、学習が大幅に遅くなる）

In [ ]:
# ログ記録目的で学習ローダーと検証ローダーの両方についてDPO損失と報酬を計算
def evaluate_dpo_loss_loader(policy_model, reference_model, train_loader, val_loader, beta, eval_iter):

    policy_model.eval()
    with torch.no_grad():
        train_loss, train_chosen_rewards, train_rejected_rewards = compute_dpo_loss_loader(
            data_loader=train_loader,
            policy_model=policy_model,
            reference_model=reference_model,
            beta=beta,
            num_batches=eval_iter
        )

        val_loss, val_chosen_rewards, val_rejected_rewards = compute_dpo_loss_loader(
            data_loader=val_loader,
            policy_model=policy_model,
            reference_model=reference_model,
            beta=beta,
            num_batches=eval_iter
        )

    res = {
        "train_loss": train_loss,
        "train_chosen_reward": train_chosen_rewards,
        "train_rejected_reward": train_rejected_rewards,
        "val_loss": val_loss,
        "val_chosen_reward": val_chosen_rewards,
        "val_rejected_reward": val_rejected_rewards
    }

    policy_model.train()
    return res

- このセクションのまとめ
    - フロー：モデル経由で`logits`を計算->ロジットから`compute_logprobs`を計算->対数確率から`compute_dpo_loss`計算
    - 上記の一連の計算プロセスを容易にする`compute_dpo_loss_batch`関数の作成
    - `compute_dpo_loss_loader`ユーティリティ関数は、`compute_dpo_loss_batch`関数をデータローダーに適用
    - `evaluate_dpo_loss_loader`関数は、ログ記録目的で学習セットと検証セットのデータローダーの両方に`compute_dpo_loss_batch`を適用する

## モデルの学習

In [ ]:
# 初期損失と報酬を表示
from previous_chapters import generate_and_print_sample

def train_model_dpo_simple(
        policy_model, reference_model, train_loader, val_loader,
        optimizer, num_epochs, beta,
        eval_freq, eval_iter, start_context, tokenizer
):

    # 損失と処理済みトークン数を追跡するリストを初期化
    tracking = {
        "train_losses": [],
        "train_chosen_rewards": [],
        "train_rejected_rewards": [],
        "val_losses": [],
        "val_chosen_rewards": [],
        "val_rejected_rewards": [],
        "tokens_seen": [],
    }
    token_seen, global_step = 0, -1

    # メインの学習ループ
    for epoch in range(num_epochs):
        policy_model.train()

        for batch in train_loader:

            optimizer.zero_grad()

            loss, chosen_rewards, rejected_rewards = compute_dpo_loss_bath(
                batch=batch,
                policy_model=policy_model,
                reference_model=reference_model,
                beta=beta
            )

            loss.backward()
            optimizer.step()

            token_seen += batch["chosen"].numel()
            global_step += 1

            # オプションの評価ステップ
            if global_step % eval_freq == 0:
                res = evaluate_dpo_loss_loader(
                    policy_model=policy_model,
                    reference_model=reference_model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    beta=beta,
                    eval_iter=eval_iter
                )
                tracking["train_losses"].append(res["train_loss"])
                tracking["train_chosen_rewards"].append(res["train_chosen_reward"])
                tracking["train_rejected_rewards"].append(res["train_rejected_reward"])
                tracking["val_losses"].append(res["val_loss"])
                tracking["val_chosen_rewards"].append(res["val_chosen_reward"])
                tracking["val_rejected_rewards"].append(res["val_rejected_reward"])
                tracking["tokens_seen"].append(token_seen)
                train_reward_margin = res["train_chosen_reward"] - res["train_rejected_reward"]
                val_reward_margin = res["val_chosen_reward"] - res["val_rejected_reward"]

                print(
                    f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {res['train_loss']:.3f}, Val loss {res['val_loss']:.3f}, "
                    f"Train reward margins {train_reward_margin:.3f}, "
                    f"Val reward margins {val_reward_margin:.3f}"
                )

        # 各エポック後にサンプルテキストを表示
        generate_and_print_sample(
            model=model,
            tokenizer=tokenizer,
            device=device,
            start_context=start_context
        )

    return tracking

In [ ]:
# 初期値確認
torch.manual_seed(123)

res = evaluate_dpo_loss_loader(
    policy_model=policy_model,
    reference_model=reference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    beta=0.1,
    eval_iter=5
)

print(f"Training loss: {res['train_loss']}")
print(f"Validation loss: {res['val_loss']}")

print(f"Train reward margin: {res['train_chosen_reward'] - res['train_rejected_reward']}")
print(f"Validation reward margin: {res['val_chosen_reward'] - res['val_chosen_reward']}")

- 初期モデル（ファインチューニングは済み）の応答をサンプル3件で確認

In [ ]:
torch.manual_seed(123)

for entry in val_data[:3]:

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("\n------------------------------------------------\n")